In [6]:
# !pip install --upgrade numba

In [7]:
# Imports
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
import shap
import matplotlib.pyplot as plt

RANDOM_STATE = 42

In [8]:
df = pd.read_csv(
    r"D:\Data_D\TMU\MRP\1.0.Dataset\Crime_Socio_Economic_Weather_Mobility.csv"
)
print("Full dataset shape:", df.shape)
print("Target distribution:\n", df["Arrest"].value_counts(normalize=True) * 100)

Full dataset shape: (7970658, 36)
Target distribution:
 Arrest
0    74.195292
1    25.804708
Name: proportion, dtype: float64


In [9]:
y_full = df["Arrest"]
X_full = df.drop(columns=["Arrest", "community_name"])  # drop target + text label

X_train_full, X_test_full, y_train_full, y_test_full = train_test_split(
    X_full,
    y_full,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_full,
)

print("Training partition shape:", X_train_full.shape)
print("Test partition shape:", X_test_full.shape)

# Keep a record of which original row indices belong to train vs. test, so the
# same partition can be reused (and verified) when the modeling dataset is
# built later in the pipeline.
train_index = X_train_full.index
test_index = X_test_full.index

Training partition shape: (6376526, 34)
Test partition shape: (1594132, 34)


In [10]:
SAMPLE_SIZE = 200_000

train_frame = X_train_full.copy()
train_frame["Arrest"] = y_train_full

sample_df = train_frame.sample(n=SAMPLE_SIZE, random_state=RANDOM_STATE)

X_sample = sample_df.drop(columns=["Arrest"])
y_sample = sample_df["Arrest"]

print("Feature-selection sample shape:", X_sample.shape)
print("Sample target distribution:\n", y_sample.value_counts(normalize=True) * 100)

Feature-selection sample shape: (200000, 34)
Sample target distribution:
 Arrest
0    74.3025
1    25.6975
Name: proportion, dtype: float64


In [11]:
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
rf.fit(X_sample, y_sample)

importance_df = pd.DataFrame(
    {"Feature": X_sample.columns, "Importance": rf.feature_importances_}
).sort_values(by="Importance", ascending=False)

pd.set_option("display.max_rows", None)
print(importance_df)

importance_df.to_csv("Feature_Importance_TrainOnly.csv", index=False)

                 Feature  Importance
7               FBI Code    0.240943
1            Description    0.129439
0           Primary Type    0.093451
2   Location Description    0.041030
18                  date    0.033236
9               Latitude    0.030902
10             Longitude    0.030082
30                   bus    0.026401
31        rail_boardings    0.026270
32           total_rides    0.026053
13                  hour    0.024868
23                  AWND    0.024339
4                   Beat    0.023382
24              temp_avg    0.022419
33                   day    0.021747
20                  TMIN    0.021649
19                  TMAX    0.021520
8                   year    0.019233
15     per_capita_income    0.018010
17     no_highschool_pct    0.016835
16     unemployment_rate    0.016768
6         community_area    0.014508
11                 month    0.014218
3               Domestic    0.012387
21                  PRCP    0.011453
12           day_of_week    0.010946
5

In [12]:
X_sub_train, X_sub_val, y_sub_train, y_sub_val = train_test_split(
    X_sample,
    y_sample,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_sample,
)

perm_result = permutation_importance(
    rf,
    X_sub_val,
    y_sub_val,
    n_repeats=5,
    random_state=RANDOM_STATE,
    scoring="f1",
)

perm_df = pd.DataFrame(
    {"Feature": X_sub_val.columns, "Importance": perm_result.importances_mean}
).sort_values(by="Importance", ascending=False)

print(perm_df)

                 Feature  Importance
7               FBI Code    0.410424
1            Description    0.190534
0           Primary Type    0.159552
2   Location Description    0.128175
18                  date    0.059653
13                  hour    0.057113
3               Domestic    0.052523
9               Latitude    0.039785
8                   year    0.032884
15     per_capita_income    0.031279
16     unemployment_rate    0.026879
17     no_highschool_pct    0.026533
10             Longitude    0.024421
4                   Beat    0.021970
31        rail_boardings    0.020686
32           total_rides    0.018613
30                   bus    0.016375
33                   day    0.013765
6         community_area    0.013081
23                  AWND    0.012892
19                  TMAX    0.009432
20                  TMIN    0.008856
24              temp_avg    0.008709
11                 month    0.007911
21                  PRCP    0.006067
5               District    0.004748
1

In [13]:
X_shap = X_sample.sample(n=100, random_state=RANDOM_STATE)

explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_shap)

print("SHAP values calculated. Shape:", np.asarray(shap_values).shape)

SHAP values calculated. Shape: (100, 34, 2)


In [14]:
# Summary plots — saved to disk so they can be embedded directly in the LaTeX chapter
shap.summary_plot(shap_values[:, :, 1], X_shap, show=False)
plt.savefig("shap_summary_beeswarm.png", bbox_inches="tight", dpi=200)
plt.close()

shap.summary_plot(shap_values[:, :, 1], X_shap, plot_type="bar", show=False)
plt.savefig("shap_summary_bar.png", bbox_inches="tight", dpi=200)
plt.close()

In [15]:
for feat in ["bus", "rail_boardings", "total_rides"]:
    shap.dependence_plot(feat, shap_values[:, :, 1], X_shap, show=False)
    plt.savefig(f"shap_dependence_{feat}.png", bbox_inches="tight", dpi=200)
    plt.close()

print("Saved SHAP plots.")

Saved SHAP plots.


In [16]:
drop_cols = [
    "is_weekend",
    "high_wind",
    "is_snow",
    "is_rain",
    "temp_category",
    "day_type",
    "SNOW",
]

lowest_seven = importance_df.tail(7)["Feature"].tolist()
print("Lowest-importance 7 features on training-only sample:", lowest_seven)

if set(lowest_seven) != set(drop_cols):
    print(
        "\n*** WARNING: the lowest-importance features on the training-only sample differ from the "
        "original drop list. Review importance_df above and update `drop_cols` before exporting "
        "the final dataset. ***"
    )
else:
    print("\nConfirmed: lowest-importance features match the original drop list.")

Lowest-importance 7 features on training-only sample: ['temp_category', 'is_rain', 'day_type', 'SNOW', 'is_weekend', 'high_wind', 'is_snow']

Confirmed: lowest-importance features match the original drop list.


In [17]:
final_predictor_cols = [c for c in X_full.columns if c not in drop_cols]

df_final = df[final_predictor_cols + ["Arrest"]].copy()

print("Final modeling dataset shape:", df_final.shape)
print("Final predictor count:", len(final_predictor_cols))
print("Final columns:", final_predictor_cols)

output_path = r"D:\Data_D\TMU\MRP\1.0.Dataset\Final_Crime_Dataset_TrainOnly.csv"
df_final.to_csv(output_path, index=False)
print("Saved final dataset to:", output_path)

Final modeling dataset shape: (7970658, 28)
Final predictor count: 27
Final columns: ['Primary Type', 'Description', 'Location Description', 'Domestic', 'Beat', 'District', 'community_area', 'FBI Code', 'year', 'Latitude', 'Longitude', 'month', 'day_of_week', 'hour', 'per_capita_income', 'unemployment_rate', 'no_highschool_pct', 'date', 'TMAX', 'TMIN', 'PRCP', 'AWND', 'temp_avg', 'bus', 'rail_boardings', 'total_rides', 'day']
Saved final dataset to: D:\Data_D\TMU\MRP\1.0.Dataset\Final_Crime_Dataset_TrainOnly.csv


In [18]:
np.save("train_index.npy", train_index.to_numpy())
np.save("test_index.npy", test_index.to_numpy())
print("Saved train/test row indices for reuse in model training.")

Saved train/test row indices for reuse in model training.
